# VisionAI Inspector — Stage 4: Defect Localization (Segmentation)

**Goal:** move from "is this image defective?" (Stages 2-3) to "*where*,
specifically, is the defect?" — using MVTec's pixel-level `ground_truth`
masks, which we have not touched at all until now.

**Protocol (unchanged):** no model here is trained on masks. Masks are used
**only at evaluation time**, to score how well a per-pixel anomaly heatmap
lines up with the real defect region. This keeps us consistent with every
earlier stage: training stays strictly normal-only.

**Two localization approaches, extending Stage 3's two models:**

- **Model A — Autoencoder pixel-level reconstruction error.** Instead of
  one MSE number per image (Stage 3), we keep the full per-pixel squared
  error map — a heatmap of exactly where reconstruction failed.
- **Model B — ResNet18 patch-level feature distance.** Instead of one
  pooled 512-dim vector per image (Stage 3), we extract a spatial feature
  map and compare each patch to the average "normal" patch at that same
  location, seen during training. This is a simplified version of
  **PaDiM**, a well-known published MVTec AD localization method (full
  PaDiM uses a per-location Gaussian + Mahalanobis distance; we simplify
  to a per-location mean + Euclidean distance to keep this tractable to
  implement and explain).

Both are evaluated against real pixel masks with **IoU**, **Dice score**,
and pixel-level precision/recall/F1 — plus a simple bounding box extracted
from each predicted mask, for a lightweight "detection" view.


## 1. Setup, imports, and loading Stage 1's splits

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from scipy import ndimage

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_utils import build_protocol_splits, make_synthetic_placeholder_dataset
from torch_dataset import BottleDataset
from autoencoder import ConvAutoencoder
from pretrained_features import build_resnet_spatial_extractor, extract_spatial_features
from evaluation import calibrate_pixel_threshold, pixel_level_report
from masks import load_mask_array, iou_score, dice_score, bounding_box_from_mask

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "mvtec_ad"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CATEGORY = "bottle"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

category_path = RAW_ROOT / CATEGORY
USING_SYNTHETIC_DATA = not category_path.exists()
if USING_SYNTHETIC_DATA:
    print("Real MVTec data not found — generating synthetic placeholder (with matching masks) for pipeline testing.")
    make_synthetic_placeholder_dataset(RAW_ROOT, category=CATEGORY, seed=RANDOM_SEED)

DATA_TAG = "[SYNTHETIC PLACEHOLDER DATA] " if USING_SYNTHETIC_DATA else ""

train_csv = PROCESSED_ROOT / f"{CATEGORY}_train.csv"
val_csv = PROCESSED_ROOT / f"{CATEGORY}_val.csv"
test_csv = PROCESSED_ROOT / f"{CATEGORY}_test.csv"

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
else:
    print("Stage 1 CSVs not found — regenerating splits with the same protocol-preserving logic.")
    train_df, val_df, test_df = build_protocol_splits(category_path, val_frac=0.15, seed=RANDOM_SEED)

print(f"\n{DATA_TAG}train (normal only): {len(train_df)}")
print(f"{DATA_TAG}val (normal only):   {len(val_df)}")
print(f"{DATA_TAG}test (untouched):    {len(test_df)}")

EVAL_SIZE = (64, 64)  # (width, height) — the common resolution every heatmap and mask is compared at

## 2. Ground truth masks — what they actually are

Every defective test image has a matching binary mask under
`ground_truth/<defect_type>/`: white pixels mark the defect, black
everywhere else. Normal (`good`) test images have no mask file at all —
`src/masks.py`'s `load_mask_array` returns an all-zero mask for those,
since there's nothing to localize.

Let's look at a few real examples before building anything.


In [ ]:
defective_examples = test_df[test_df["label"] == "defective"].groupby("defect_type").first().reset_index()

fig, axes = plt.subplots(2, len(defective_examples), figsize=(3.5 * len(defective_examples), 7))
for col, row in defective_examples.iterrows():
    from PIL import Image
    img = np.array(Image.open(row["filepath"]).convert("RGB").resize(EVAL_SIZE))
    mask = load_mask_array(row["filepath"], EVAL_SIZE)

    ax_img = axes[0, col] if len(defective_examples) > 1 else axes[0]
    ax_mask = axes[1, col] if len(defective_examples) > 1 else axes[1]
    ax_img.imshow(img); ax_img.set_title(f"{DATA_TAG}{row['defect_type']}"); ax_img.axis("off")
    ax_mask.imshow(mask, cmap="gray"); ax_mask.set_title("ground truth mask"); ax_mask.axis("off")

plt.tight_layout()
plt.show()

## 3. Model A — Autoencoder pixel-level heatmap

We reuse the exact autoencoder trained in Stage 3 (loading its saved
weights rather than retraining — a model, once trained, doesn't need to be
retrained every time you want to use it, which is the entire point of
saving weights to disk). If Stage 3 hasn't been run, we train a quick
fallback copy here so this notebook still works standalone.


In [ ]:
ae_transform = transforms.Compose([
    transforms.Resize(EVAL_SIZE),
    transforms.ToTensor(),
])

autoencoder = ConvAutoencoder(latent_dim=64, dropout_p=0.1).to(DEVICE)
weights_path = PROCESSED_ROOT / f"{CATEGORY}_autoencoder_weights.pt"

if weights_path.exists():
    autoencoder.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    autoencoder.eval()
    print(f"Loaded trained autoencoder weights from Stage 3 -> {weights_path}")
else:
    print("Stage 3 autoencoder weights not found — training a quick fallback copy here.")
    train_loader_fallback = DataLoader(BottleDataset(train_df, transform=ae_transform), batch_size=16, shuffle=True)
    optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    autoencoder.train()
    for epoch in range(20):
        for images, _ in train_loader_fallback:
            images = images.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(autoencoder(images), images)
            loss.backward()
            optimizer.step()
    autoencoder.eval()
    print("Fallback training complete.")

In [ ]:
def compute_pixel_error_maps(dataframe, model, transform):
    """Per-image, per-pixel squared error between input and reconstruction, averaged over channels."""
    dataset = BottleDataset(dataframe, transform=transform)
    loader = DataLoader(dataset, batch_size=16, shuffle=False)
    maps = []
    model.eval()
    with torch.no_grad():
        for images, _labels in loader:
            images = images.to(DEVICE)
            recon = model(images)
            error_map = ((recon - images) ** 2).mean(dim=1)  # average over channels -> (batch, H, W)
            maps.append(error_map.cpu().numpy())
    return np.concatenate(maps, axis=0)

val_heatmaps_ae = compute_pixel_error_maps(val_df, autoencoder, ae_transform)
test_heatmaps_ae = compute_pixel_error_maps(test_df, autoencoder, ae_transform)

print(f"{DATA_TAG}val_heatmaps_ae shape: {val_heatmaps_ae.shape}")
print(f"{DATA_TAG}test_heatmaps_ae shape: {test_heatmaps_ae.shape}")

## 4. Model B — ResNet18 patch-level feature distance

See `src/pretrained_features.py`'s `build_resnet_spatial_extractor` for the
full reasoning (a simplified PaDiM). We compute one "average normal patch"
feature map from the training set, then measure how far each test patch's
features are from that average at the same spatial location.


In [ ]:
resnet_spatial, weights_are_pretrained, resnet_preprocess = build_resnet_spatial_extractor(DEVICE)
short_tag = "" if weights_are_pretrained else "[RANDOM-INIT FALLBACK, NOT TRANSFER LEARNING] "

train_loader_resnet = DataLoader(BottleDataset(train_df, transform=resnet_preprocess), batch_size=16, shuffle=False)
val_loader_resnet = DataLoader(BottleDataset(val_df, transform=resnet_preprocess), batch_size=16, shuffle=False)
test_loader_resnet = DataLoader(BottleDataset(test_df, transform=resnet_preprocess), batch_size=16, shuffle=False)

train_features_spatial = extract_spatial_features(resnet_spatial, train_loader_resnet, DEVICE)  # (n, 128, 28, 28)
val_features_spatial = extract_spatial_features(resnet_spatial, val_loader_resnet, DEVICE)
test_features_spatial = extract_spatial_features(resnet_spatial, test_loader_resnet, DEVICE)

print(f"{short_tag}{DATA_TAG}train_features_spatial shape: {train_features_spatial.shape}")

# The "average normal patch" reference: mean feature vector at each spatial location, across all training images.
normal_reference_map = train_features_spatial.mean(axis=0)  # (128, 28, 28)
print(f"{short_tag}{DATA_TAG}normal_reference_map shape: {normal_reference_map.shape}")

In [ ]:
def compute_resnet_distance_maps(features_spatial, reference_map, output_size):
    """Per-location Euclidean distance to the normal reference, then upsample to output_size."""
    diff = features_spatial - reference_map[np.newaxis, :, :, :]   # (n, 128, 28, 28)
    distance_map = np.sqrt((diff ** 2).sum(axis=1))                 # (n, 28, 28) -- L2 distance over channels
    zoom_factors = (1, output_size[1] / distance_map.shape[1], output_size[0] / distance_map.shape[2])
    upsampled = ndimage.zoom(distance_map, zoom_factors, order=1)   # bilinear-like upsampling to EVAL_SIZE
    return upsampled

val_heatmaps_resnet = compute_resnet_distance_maps(val_features_spatial, normal_reference_map, EVAL_SIZE)
test_heatmaps_resnet = compute_resnet_distance_maps(test_features_spatial, normal_reference_map, EVAL_SIZE)

print(f"{short_tag}{DATA_TAG}test_heatmaps_resnet shape: {test_heatmaps_resnet.shape}")

## 5. Threshold calibration and building the ground truth mask stack

Same discipline as every prior stage: the pixel-level threshold for each
model is calibrated only from **validation** heatmaps (normal images), and
we never look at test masks until the final evaluation step below.


In [ ]:
PIXEL_THRESHOLD_PERCENTILE = 5

ae_pixel_threshold = calibrate_pixel_threshold(val_heatmaps_ae, PIXEL_THRESHOLD_PERCENTILE)
resnet_pixel_threshold = calibrate_pixel_threshold(val_heatmaps_resnet, PIXEL_THRESHOLD_PERCENTILE)

print(f"{DATA_TAG}Autoencoder pixel threshold: {ae_pixel_threshold:.6f}")
print(f"{short_tag}{DATA_TAG}ResNet patch-distance pixel threshold: {resnet_pixel_threshold:.6f}")

pred_masks_ae = (test_heatmaps_ae > ae_pixel_threshold).astype(np.uint8)
pred_masks_resnet = (test_heatmaps_resnet > resnet_pixel_threshold).astype(np.uint8)

true_masks = np.stack([load_mask_array(fp, EVAL_SIZE) for fp in test_df["filepath"]])
print(f"{DATA_TAG}true_masks shape: {true_masks.shape}")

## 6. Evaluation: IoU, Dice, and pixel-level precision/recall/F1

IoU and Dice are computed **per defective image only** (they're undefined
for a genuinely empty ground truth mask, since there's no defect region to
overlap with) and then averaged. Pixel-level precision/recall/F1 use every
pixel from every test image, including `good` ones — this is the strictest
evaluation, since a single stray falsely-flagged patch on a normal image
hurts precision here even if that image's overall classification (Stage 3)
was correct.


In [ ]:
def summarize_localization(name, pred_masks, true_masks, defect_mask_rows, tag=""):
    ious, dices = [], []
    for i in defect_mask_rows:
        ious.append(iou_score(pred_masks[i], true_masks[i]))
        dices.append(dice_score(pred_masks[i], true_masks[i]))
    mean_iou = np.nanmean(ious)
    mean_dice = np.nanmean(dices)
    print(f"=== {tag}{name} ===")
    print(f"Mean IoU (defective images only):  {mean_iou:.4f}")
    print(f"Mean Dice (defective images only): {mean_dice:.4f}")
    pixel_level_report(true_masks, pred_masks)
    return {"mean_iou": mean_iou, "mean_dice": mean_dice}

defective_rows = test_df.index[test_df["label"] == "defective"].tolist()

ae_localization_result = summarize_localization("Autoencoder heatmap", pred_masks_ae, true_masks, defective_rows, tag=DATA_TAG)
print()
resnet_localization_result = summarize_localization("ResNet patch-distance heatmap", pred_masks_resnet, true_masks, defective_rows, tag=short_tag + DATA_TAG)

## 7. Visualize: image, ground truth, heatmap, and predicted mask side by side

This is the real payoff of Stage 4 — actually seeing whether the highlighted region matches where the defect is.

In [ ]:
def show_localization(indices, heatmaps, pred_masks, title):
    fig, axes = plt.subplots(4, len(indices), figsize=(3.2 * len(indices), 12))
    for col, idx in enumerate(indices):
        from PIL import Image
        row = test_df.iloc[idx]
        img = np.array(Image.open(row["filepath"]).convert("RGB").resize(EVAL_SIZE))

        axes[0, col].imshow(img); axes[0, col].set_title(row["defect_type"]); axes[0, col].axis("off")
        axes[1, col].imshow(true_masks[idx], cmap="gray"); axes[1, col].set_title("ground truth"); axes[1, col].axis("off")
        axes[2, col].imshow(heatmaps[idx], cmap="inferno"); axes[2, col].set_title("heatmap"); axes[2, col].axis("off")
        axes[3, col].imshow(pred_masks[idx], cmap="gray"); axes[3, col].set_title("predicted mask"); axes[3, col].axis("off")

    fig.suptitle(f"{DATA_TAG}{title}", y=1.02)
    plt.tight_layout()
    plt.show()

show_indices = defective_rows[:4] if len(defective_rows) >= 4 else defective_rows
show_localization(show_indices, test_heatmaps_ae, pred_masks_ae, "Model A: Autoencoder localization")
show_localization(show_indices, test_heatmaps_resnet, pred_masks_resnet, f"{short_tag}Model B: ResNet patch-distance localization")

## 8. Bounding boxes from predicted masks — a lightweight 'detection' view

Drawing a box around the largest connected predicted-anomalous region turns a segmentation mask into something closer to an object-detection output, with no separate detection model needed.

In [ ]:
fig, axes = plt.subplots(1, len(show_indices), figsize=(3.5 * len(show_indices), 4))
for col, idx in enumerate(show_indices):
    from PIL import Image
    row = test_df.iloc[idx]
    img = np.array(Image.open(row["filepath"]).convert("RGB").resize(EVAL_SIZE))
    ax = axes[col] if len(show_indices) > 1 else axes
    ax.imshow(img)
    ax.axis("off")

    pred_box = bounding_box_from_mask(pred_masks_ae[idx])
    true_box = bounding_box_from_mask(true_masks[idx])
    if pred_box is not None:
        x0, y0, x1, y1 = pred_box
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="red", facecolor="none", label="predicted"))
    if true_box is not None:
        x0, y0, x1, y1 = true_box
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="lime", facecolor="none", label="ground truth"))
    ax.set_title(row["defect_type"], fontsize=9)

fig.suptitle(f"{DATA_TAG}Autoencoder: predicted (red) vs ground truth (green) bounding boxes", y=1.05)
plt.tight_layout()
plt.show()

## 9. Model comparison

In [ ]:
comparison_df = pd.DataFrame([
    {"model": "Autoencoder heatmap", "mean_iou": ae_localization_result["mean_iou"], "mean_dice": ae_localization_result["mean_dice"]},
    {"model": f"{short_tag}ResNet patch-distance", "mean_iou": resnet_localization_result["mean_iou"], "mean_dice": resnet_localization_result["mean_dice"]},
])
print(f"{DATA_TAG}")
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(comparison_df))
width = 0.35
ax.bar(x - width/2, comparison_df["mean_iou"], width, label="Mean IoU")
ax.bar(x + width/2, comparison_df["mean_dice"], width, label="Mean Dice")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["model"], rotation=10, ha="right")
ax.set_title(f"{DATA_TAG}Stage 4 — localization quality comparison")
ax.legend()
fig.tight_layout(pad=2.0)
plt.show()

## 10. Save Stage 4 outputs

In [ ]:
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
np.savez(
    PROCESSED_ROOT / f"{CATEGORY}_stage4_heatmaps.npz",
    test_heatmaps_ae=test_heatmaps_ae,
    test_heatmaps_resnet=test_heatmaps_resnet,
    true_masks=true_masks,
)
comparison_df.to_csv(PROCESSED_ROOT / f"{CATEGORY}_stage4_comparison.csv", index=False)
print(f"Saved heatmaps/masks -> {PROCESSED_ROOT / f'{CATEGORY}_stage4_heatmaps.npz'}")
print(f"Saved comparison table -> {PROCESSED_ROOT / f'{CATEGORY}_stage4_comparison.csv'}")

print(f"\n{'SYNTHETIC PLACEHOLDER outputs saved.' if USING_SYNTHETIC_DATA else 'Real dataset outputs saved.'}")
if not weights_are_pretrained:
    print("NOTE: ResNet results used RANDOM-INIT fallback weights -- pipeline smoke-test only.")

## 11. Summary — Stage 4 complete

What we built:

- Real pixel-level ground truth understanding (`src/masks.py`), with no
  training ever performed on masks — protocol preserved end to end.
- **Model A**: reused Stage 3's trained autoencoder to produce full
  per-pixel reconstruction-error heatmaps instead of one number per image.
- **Model B**: a simplified PaDiM-style approach — per-location ResNet
  features compared against a per-location "normal average" from training,
  producing a patch-level anomaly heatmap.
- Evaluation with IoU, Dice, and strict pixel-level precision/recall/F1.
- Bounding boxes derived directly from predicted masks — a detection-style
  view with no separate detection model.

**Next (Stage 5):** Vision Transformers — comparing a ViT-based approach
against the CNN-based methods built so far, on accuracy, training
requirements, inference latency, and model size.
